# evaluation

Evaluates one checkpoint on the held-out test split and writes a `results/*.json`.

Two variants, switched by `EVAL_BASE` in the config cell:
- `EVAL_BASE=False` (default) loads the **trained** weights from
  `PROJECT_ROOT/models/<model>_seed<seed>/` and writes `results/<model>_seed<seed>.json`.
- `EVAL_BASE=True` loads the **untrained base model** (the scaling-curve baseline) and writes
  `results/<model>_base.json`.

Both variants read the prompt template from the trained checkpoint's `run_config.json`, so the
base baseline uses the **same prompt** as training and the base-vs-trained delta isolates
training, not a prompt change. `GEN_MAX_NEW` is a **frozen** study constant — derived once from
the real test set, hardcoded, identical across every size, seed, and variant.

Reports a discontinuous metric (exact match) and a continuous one (token-sim), plus two
diagnostics: EOS-emission rate (% that stop on their own) and length ratio (pred/ref token
length). Aggregate `results/*.json` across sizes/seeds/variants in a separate plotting step.

In [ ]:
# === config ===
import os, json, math, difflib
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- which checkpoint to evaluate ----
MODEL_NAME = "Qwen2.5-Coder-0.5B"    # the RUN_NAME stem used in training
SEED       = 0

# ---- base vs trained ----
# EVAL_BASE=True evaluates the UNTRAINED base model as the scaling-curve baseline. It still uses
# the SAME prompt the checkpoint was trained with (loaded in the next cell), so the
# base-vs-trained delta isolates training and is not confounded by a prompt change.
EVAL_BASE     = False
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B"   # HF id of the base; must be MODEL_NAME's actual base

PROJECT_ROOT_OVERRIDE = None
def find_project_root(markers=(".git", "pyproject.toml", "setup.py")):
    p = Path(os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if any((cand / m).exists() for m in markers):
            return cand
    return p
PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else find_project_root()
MODEL_DIR    = PROJECT_ROOT / "models" / f"{MODEL_NAME}_seed{SEED}"
RESULTS_DIR  = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# what to load, what variant to tag, where to write
VARIANT     = "base" if EVAL_BASE else "trained"
LOAD_FROM   = BASE_MODEL_ID if EVAL_BASE else str(MODEL_DIR)
RESULT_STEM = f"{MODEL_NAME}_base" if EVAL_BASE else f"{MODEL_NAME}_seed{SEED}"

# ---- FROZEN eval constants: identical across EVERY size, seed, AND variant ----
# Derive GEN_MAX_NEW ONCE from the real test set, then hardcode. Recomputing per run would
# make it differ by eval set and break cross-size comparability.
GEN_MAX_NEW = 5632          # p99 of the 68-example test set, rounded up to 256
EVAL_BATCH  = 8             # lower to 2-4 at 7B
MAX_EVAL    = None          # None = full test split

REPO     = "notoh/exebench_llvm_o3"
REVISION = "4d26c13bf8473067db611fce495ce4dd27639c0a"

device = "cuda" if torch.cuda.is_available() else "cpu"
# trained eval REQUIRES the checkpoint weights; base eval only needs MODEL_DIR for the prompt
# (it warns and falls back to the default prompt if MODEL_DIR is absent).
if not EVAL_BASE:
    assert MODEL_DIR.exists(), f"no checkpoint at {MODEL_DIR} -- train it first"
print("device   :", device)
print("variant  :", VARIANT)
print("load from:", LOAD_FROM)
print("result   :", RESULT_STEM + ".json")

In [ ]:
# === load model + the prompt it was trained with ===
# Prompt comparability is load-bearing: the base baseline MUST use the SAME prompt as the trained
# checkpoint, or the base-vs-trained delta is confounded by the prompt. So read the template from
# the checkpoint's run_config in BOTH variants.
cfg = {}
cfg_path = MODEL_DIR / "run_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
elif EVAL_BASE:
    print("WARNING: no run_config.json at", MODEL_DIR)
    print("         base eval is falling back to the DEFAULT prompt, which may NOT match the")
    print("         trained checkpoint's prompt. Train the checkpoint first so base and trained")
    print("         share a prompt, or the base-vs-trained delta is not interpretable.")
PROMPT_TMPL = cfg.get("prompt_tmpl", "Compile the following C to LLVM IR at -O3:\n{c}\n---\n")
print("prompt template:", repr(PROMPT_TMPL))

tok = AutoTokenizer.from_pretrained(LOAD_FROM)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LOAD_FROM, dtype=torch.bfloat16 if device == "cuda" else torch.float32)
model.to(device).eval()
model.config.use_cache = True          # fast generation

test_ds = load_dataset(REPO, revision=REVISION)["test"]
if MAX_EVAL:
    test_ds = test_ds.select(range(min(MAX_EVAL, len(test_ds))))
print(f"eval examples: {len(test_ds)}")

In [ ]:
# === sanity: does the FROZEN GEN_MAX_NEW cover the targets? (warn only -- never auto-change) ===
tgt_lens = [len(tok(r["o3_ir"], add_special_tokens=False)["input_ids"]) for r in test_ds]
p50, p95, p99 = (int(np.percentile(tgt_lens, q)) for q in (50, 95, 99))
print(f"target tokens  p50={p50}  p95={p95}  p99={p99}  max={max(tgt_lens)}")
over = sum(l > GEN_MAX_NEW for l in tgt_lens)
print(f"GEN_MAX_NEW={GEN_MAX_NEW}  ({over}/{len(tgt_lens)} targets exceed it)")
if GEN_MAX_NEW < p99:
    print("WARNING: ceiling < p99 -- long targets truncate. Keep it frozen anyway for comparability.")

In [ ]:
# === metrics + batched eval ===
def normalize(s):
    return "\n".join(l.rstrip() for l in s.strip().splitlines())

def token_sim(a, b):           # continuous metric in [0, 1]
    ta = tok(a, add_special_tokens=False)["input_ids"]
    tb = tok(b, add_special_tokens=False)["input_ids"]
    return difflib.SequenceMatcher(None, ta, tb).ratio()

tok.padding_side = "left"      # correct decoder-only batched generation
exact, sims, len_ratio, eos_emitted = 0, [], [], 0
n = len(test_ds)
for start in range(0, n, EVAL_BATCH):
    recs    = [test_ds[k] for k in range(start, min(start + EVAL_BATCH, n))]
    prompts = [PROMPT_TMPL.format(c=r["c_source"]) for r in recs]
    enc = tok(prompts, add_special_tokens=False, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW,
                             do_sample=False, pad_token_id=tok.pad_token_id)
    new = gen[:, enc["input_ids"].shape[1]:]
    for j, r in enumerate(recs):
        ids_out = new[j]
        if (ids_out == tok.eos_token_id).any():
            eos_emitted += 1
        pred = tok.decode(ids_out, skip_special_tokens=True)
        ref  = r["o3_ir"]
        if normalize(pred) == normalize(ref):
            exact += 1
        sims.append(token_sim(pred, ref))
        rl = len(tok(pred, add_special_tokens=False)["input_ids"])
        tl = len(tok(ref,  add_special_tokens=False)["input_ids"])
        len_ratio.append(rl / max(tl, 1))
    print(f"  {min(start + EVAL_BATCH, n)}/{n} done")

results = {
    "model": MODEL_NAME, "variant": VARIANT, "seed": (None if EVAL_BASE else SEED), "n": n,
    "exact_match":    exact / n,                 # discontinuous
    "token_sim_mean": float(np.mean(sims)),      # continuous
    "token_sim_std":  float(np.std(sims)),
    "eos_rate":       eos_emitted / n,           # % that stopped on their own
    "len_ratio_mean": float(np.mean(len_ratio)), # 1.0 = right length; >>1 = over-generating
    "gen_max_new":    GEN_MAX_NEW,
}
print("\n=== results ===")
for k, v in results.items():
    print(f"  {k:16s}: {v}")

out = RESULTS_DIR / f"{RESULT_STEM}.json"
out.write_text(json.dumps(results, indent=2))
print("\nsaved ->", out)